# NB03 — SQL Analytical Foundation
**Signal/Pulse · Pure SQL Analysis**

**Purpose:** Demonstrate serious analytical SQL on the ingested database.
All analytical logic lives in SQL strings executed via `sqlite3`.
Pandas is used only for display and visualisation — not for computation.

**SQL skills demonstrated:**
- CTEs (Common Table Expressions)
- Window functions: `RANK()`, `ROW_NUMBER()`, `LAG()`, `AVG() OVER()`
- Self-JOINs for reviewer cross-category behaviour
- Subqueries and `EXISTS`
- `GROUP BY` with `HAVING`, `CASE` expressions
- Date functions for time-series aggregation

**This notebook is the primary SQL portfolio piece.**


## 0. Setup

In [1]:
import sys
sys.path.insert(0, "..")

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from src.schema import get_connection

conn = get_connection()

# Confirm data is present
for tbl in ['reviews', 'trends_weekly', 'yt_comments', 'products']:
    n = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl:<20}: {n:>8,} rows')


  reviews             :   22,451 rows
  trends_weekly       :    4,635 rows
  yt_comments         :   60,676 rows
  products            :   31,355 rows


## 1. Review Volume by Category Over Time

**SQL pattern:** `GROUP BY review_year, review_month, category_id` with a CTE
to compute monthly totals, then a second CTE for running cumulative totals.
Window functions: `SUM() OVER (PARTITION BY ... ORDER BY ...)`.


In [2]:
sql_vol = '''
WITH monthly AS (
    SELECT
        r.review_year,
        r.review_month,
        c.normalized_name       AS category,
        c.tier,
        COUNT(*)                AS review_count,
        ROUND(AVG(r.rating), 2) AS avg_rating
    FROM reviews r
    JOIN categories c ON r.category_id = c.category_id
    WHERE r.review_year BETWEEN 2020 AND 2025
    GROUP BY r.review_year, r.review_month, c.category_id
),
cumulative AS (
    SELECT
        *,
        SUM(review_count) OVER (
            PARTITION BY category
            ORDER BY review_year, review_month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_count
    FROM monthly
)
SELECT * FROM cumulative
ORDER BY review_year, review_month, category
'''

df_vol = pd.read_sql(sql_vol, conn)
print(f'Rows: {len(df_vol)}')
print(df_vol.head(20).to_string(index=False))


Rows: 557
 review_year  review_month       category      tier  review_count  avg_rating  cumulative_count
        2020             1       emulsion  skincare             7        4.05                 7
        2020             1     eye_shadow cosmetics             3        3.22                 3
        2020             1     face_cream  skincare            10        4.13                10
        2020             1      face_wash  skincare            15        3.58                15
        2020             1     foundation cosmetics             2        3.67                 2
        2020             1  serum_essence  skincare             1        3.67                 1
        2020             1 sun_protection  skincare             1        3.67                 1
        2020             1   toner_lotion  skincare            10        4.13                10
        2020             2     eye_shadow cosmetics             1        1.00                 4
        2020             2    

## 2. Rating Distributions by Category and Tier

**SQL pattern:** `GROUP BY` with `HAVING`, `CASE` expressions for rating buckets.
Shows how the skincare/cosmetics split looks in terms of consumer satisfaction.


In [3]:
sql_ratings = '''
SELECT
    c.normalized_name                    AS category,
    c.tier,
    COUNT(*)                             AS total_reviews,
    ROUND(AVG(r.rating), 2)             AS avg_rating,
    ROUND(MIN(r.rating), 2)             AS min_rating,
    ROUND(MAX(r.rating), 2)             AS max_rating,
    SUM(CASE WHEN r.rating >= 4.0 THEN 1 ELSE 0 END) AS high_rating_count,
    SUM(CASE WHEN r.rating <  3.0 THEN 1 ELSE 0 END) AS low_rating_count,
    ROUND(
        100.0 * SUM(CASE WHEN r.rating >= 4.0 THEN 1 ELSE 0 END) / COUNT(*),
        1
    ) AS pct_high_rating
FROM reviews r
JOIN categories c ON r.category_id = c.category_id
WHERE r.rating IS NOT NULL
GROUP BY c.category_id
HAVING COUNT(*) >= 10
ORDER BY avg_rating DESC
'''

df_ratings = pd.read_sql(sql_ratings, conn)
print('Rating distributions by category:')
print(df_ratings.to_string(index=False))


Rating distributions by category:
      category      tier  total_reviews  avg_rating  min_rating  max_rating  high_rating_count  low_rating_count  pct_high_rating
 serum_essence  skincare            891        3.91        0.33         5.0                447                55             50.2
     face_wash  skincare           4428        3.90        0.33         5.0               2226               357             50.3
      emulsion  skincare           4295        3.89        0.33         5.0               2133               282             49.7
    lip_colour cosmetics           2224        3.88        0.33         5.0               1102               179             49.6
sun_protection  skincare           1113        3.83        0.33         5.0                503                83             45.2
    face_cream  skincare           2327        3.81        0.33         5.0               1016               175             43.7
    foundation cosmetics           1329        3.80     

## 3. Product Concentration — Top N Products by Review Volume

**SQL pattern:** CTE + `RANK()` window function over review count per tier.
Uses the `products` table directly since `reviews.brand_id` is a post-hoc lens
applied in NB04 — ingestion is category-driven, not brand-driven.

Shows which products dominate each category tier by consumer attention.


In [4]:
sql_products = '''
WITH product_stats AS (
    SELECT
        p.source_item_id,
        p.product_name,
        c.tier,
        c.normalized_name    AS category,
        p.review_count       AS source_review_count,
        p.review_avg         AS source_avg_rating,
        s.source_name
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    JOIN sources    s ON p.source_id    = s.source_id
    WHERE p.review_count IS NOT NULL
      AND COALESCE(p.tier_predicted, p.tier_override, c.tier) IN ('skincare', 'cosmetics')
),
ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY tier ORDER BY source_review_count DESC) AS tier_rank
    FROM product_stats
)
SELECT *
FROM ranked
WHERE tier_rank <= 10
ORDER BY tier, tier_rank
'''

df_products = pd.read_sql(sql_products, conn)
print('Top 10 products by review count per tier:')
print(df_products.to_string(index=False))


Top 10 products by review count per tier:
     source_item_id                                                                                                                                         product_name      tier       category  source_review_count  source_avg_rating source_name  tier_rank
           10146071                                                                                                                                                 マスカラ cosmetics     foundation             23565869                NaN       cosme          1
           10054624                                                                                                                                             フェイスパウダー cosmetics     lip_colour             23565869                NaN       cosme          1
           10248076                                                                                                                                             フェイスパウダー cosmetics     lip_

## 4. Reviewer Behaviour — Cross-Category Exploration

**SQL pattern:** Self-JOIN on `reviews` via `reviewer_id`. Find reviewers
who have reviewed both skincare AND cosmetics products — the 'switchers'
whose language may carry signals about the category shift.

**This is the query that justifies the separate reviewers table design.**


In [5]:
sql_crosscat = '''
WITH skin_agg AS (
    SELECT
        r.reviewer_id,
        COUNT(*)              AS skincare_reviews,
        ROUND(AVG(r.rating), 2) AS avg_skincare_rating,
        MIN(r.review_date)    AS first_skincare_review
    FROM reviews r
    JOIN categories c ON r.category_id = c.category_id
    WHERE c.tier = 'skincare'
      AND r.reviewer_id IS NOT NULL
    GROUP BY r.reviewer_id
    HAVING COUNT(*) >= 2
),
cosm_agg AS (
    SELECT
        r.reviewer_id,
        COUNT(*)              AS cosmetics_reviews,
        ROUND(AVG(r.rating), 2) AS avg_cosmetics_rating,
        MIN(r.review_date)    AS first_cosmetics_review
    FROM reviews r
    JOIN categories c ON r.category_id = c.category_id
    WHERE c.tier = 'cosmetics'
      AND r.reviewer_id IS NOT NULL
    GROUP BY r.reviewer_id
    HAVING COUNT(*) >= 2
)
SELECT
    s.reviewer_id,
    s.skincare_reviews,
    c.cosmetics_reviews,
    s.avg_skincare_rating,
    c.avg_cosmetics_rating,
    s.first_skincare_review,
    c.first_cosmetics_review
FROM skin_agg s
JOIN cosm_agg c USING (reviewer_id)
ORDER BY (s.skincare_reviews + c.cosmetics_reviews) DESC
LIMIT 50
'''

df_cross = pd.read_sql(sql_crosscat, conn)
print(f'Cross-category reviewers: {len(df_cross)}')
if not df_cross.empty:
    print(df_cross.head(15).to_string(index=False))
    print(f'\nAvg skincare rating  (cross-cat reviewers): {df_cross["avg_skincare_rating"].mean():.2f}')
    print(f'Avg cosmetics rating (cross-cat reviewers): {df_cross["avg_cosmetics_rating"].mean():.2f}')


Cross-category reviewers: 10
 reviewer_id  skincare_reviews  cosmetics_reviews  avg_skincare_rating  avg_cosmetics_rating first_skincare_review first_cosmetics_review
        3364                 3                  2                 3.67                  2.67            2025-10-07             2026-01-11
        3588                 3                  2                 4.78                  4.00            2024-07-11             2023-07-07
        6046                 2                  3                 3.67                  4.11            2025-08-26             2024-08-24
        2739                 2                  2                 3.67                  3.67            2025-12-17             2025-12-01
        2807                 2                  2                 4.67                  3.67            2025-09-07             2021-11-29
        2918                 2                  2                 4.33                  4.33            2024-06-03             2024-01-09
     

## 5. Year-over-Year Review Volume Change

**SQL pattern:** `LAG()` window function to compare each year's review count
to the prior year. The growth rate quantifies the structural shift hypothesis.


In [6]:
sql_yoy = '''
WITH yearly AS (
    SELECT
        c.tier,
        r.review_year,
        COUNT(*) AS review_count
    FROM reviews r
    JOIN categories c ON r.category_id = c.category_id
    WHERE r.review_year BETWEEN 2020 AND 2025
    GROUP BY c.tier, r.review_year
),
with_lag AS (
    SELECT
        tier,
        review_year,
        review_count,
        LAG(review_count) OVER (
            PARTITION BY tier
            ORDER BY review_year
        ) AS prev_year_count
    FROM yearly
)
SELECT
    tier,
    review_year,
    review_count,
    prev_year_count,
    ROUND(
        100.0 * (review_count - prev_year_count) / NULLIF(prev_year_count, 0),
        1
    ) AS yoy_pct_change
FROM with_lag
ORDER BY tier, review_year
'''

df_yoy = pd.read_sql(sql_yoy, conn)
print('Year-over-year review volume change by tier:')
print(df_yoy.to_string(index=False))


Year-over-year review volume change by tier:
     tier  review_year  review_count  prev_year_count  yoy_pct_change
cosmetics         2020            68              NaN             NaN
cosmetics         2021           152             68.0           123.5
cosmetics         2022           337            152.0           121.7
cosmetics         2023           566            337.0            68.0
cosmetics         2024          1024            566.0            80.9
cosmetics         2025          1459           1024.0            42.5
 skincare         2020           343              NaN             NaN
 skincare         2021           617            343.0            79.9
 skincare         2022           618            617.0             0.2
 skincare         2023          1018            618.0            64.7
 skincare         2024          2069           1018.0           103.2
 skincare         2025          6933           2069.0           235.1


## 6. Seasonal Patterns in Review Activity

**SQL pattern:** `GROUP BY review_month` with `AVG()`.
Uses `CASE` to label Q1–Q4. Reveals whether skincare peaks in autumn/winter
(consistent with Japanese 乾燥 season narrative).


In [7]:
sql_seasonal = '''
SELECT
    c.tier,
    r.review_month,
    CASE
        WHEN r.review_month IN (1,2,3)   THEN 'Q1 (Jan-Mar)'
        WHEN r.review_month IN (4,5,6)   THEN 'Q2 (Apr-Jun)'
        WHEN r.review_month IN (7,8,9)   THEN 'Q3 (Jul-Sep)'
        ELSE                                  'Q4 (Oct-Dec)'
    END AS quarter,
    COUNT(*) AS review_count,
    ROUND(AVG(r.rating), 2) AS avg_rating
FROM reviews r
JOIN categories c ON r.category_id = c.category_id
WHERE r.review_year BETWEEN 2020 AND 2024
GROUP BY c.tier, r.review_month
ORDER BY c.tier, r.review_month
'''

df_seasonal = pd.read_sql(sql_seasonal, conn)
print('Seasonal review patterns by tier:')
print(df_seasonal.to_string(index=False))


Seasonal review patterns by tier:
     tier  review_month      quarter  review_count  avg_rating
cosmetics             1 Q1 (Jan-Mar)           228        3.85
cosmetics             2 Q1 (Jan-Mar)           191        3.89
cosmetics             3 Q1 (Jan-Mar)           173        3.70
cosmetics             4 Q2 (Apr-Jun)           142        3.56
cosmetics             5 Q2 (Apr-Jun)           135        3.96
cosmetics             6 Q2 (Apr-Jun)           163        3.58
cosmetics             7 Q3 (Jul-Sep)           120        3.77
cosmetics             8 Q3 (Jul-Sep)           100        3.84
cosmetics             9 Q3 (Jul-Sep)           143        3.78
cosmetics            10 Q4 (Oct-Dec)           116        3.85
cosmetics            11 Q4 (Oct-Dec)           198        3.80
cosmetics            12 Q4 (Oct-Dec)           438        3.82
 skincare             1 Q1 (Jan-Mar)           315        3.91
 skincare             2 Q1 (Jan-Mar)           234        3.82
 skincare            

## 7. Category Co-occurrence — Existence Test

**SQL pattern:** Subquery with `EXISTS`. Finds categories that share
reviewers with skincare. Establishes the connection between categories
and quantifies how much the skincare audience overlaps with other tiers.


In [8]:
sql_exist = '''
SELECT
    c2.normalized_name         AS other_category,
    c2.tier,
    COUNT(DISTINCT r2.reviewer_id) AS shared_reviewers
FROM reviews r2
JOIN categories c2 ON r2.category_id = c2.category_id
WHERE c2.tier != 'skincare'
  AND EXISTS (
      SELECT 1
      FROM reviews r1
      JOIN categories c1 ON r1.category_id = c1.category_id
      WHERE c1.tier = 'skincare'
        AND r1.reviewer_id = r2.reviewer_id
        AND r1.reviewer_id IS NOT NULL
  )
GROUP BY c2.category_id
ORDER BY shared_reviewers DESC
'''

df_cooc = pd.read_sql(sql_exist, conn)
print('Categories with shared reviewers vs skincare:')
print(df_cooc.to_string(index=False))


Categories with shared reviewers vs skincare:
other_category      tier  shared_reviewers
    lip_colour cosmetics               324
    foundation cosmetics               166
    eye_shadow cosmetics               150


## 8. Google Trends — Search Velocity (LAG)

**SQL pattern:** `LAG()` on trends_weekly to compute week-over-week change.
A rolling 4-week average smooths noise.

Block A = unanchored absolute interest indices.
Block B = normalized relative to スキンケア anchor.


In [9]:
sql_trend_vel = '''
WITH weekly AS (
    SELECT
        term,
        term_group,
        week_start,
        interest,
        LAG(interest, 1) OVER (PARTITION BY term, term_group ORDER BY week_start) AS prev_week,
        AVG(interest) OVER (
            PARTITION BY term, term_group
            ORDER BY week_start
            ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
        ) AS rolling_4wk_avg
    FROM trends_weekly
    WHERE term_group = 'block_A'
)
SELECT
    term,
    week_start,
    interest,
    prev_week,
    (interest - prev_week) AS wk_delta,
    ROUND(rolling_4wk_avg, 1) AS rolling_avg
FROM weekly
WHERE week_start >= '2020-01-01'
ORDER BY term, week_start
'''

df_vel = pd.read_sql(sql_trend_vel, conn)
print(f'Trend velocity rows: {len(df_vel)}')
if not df_vel.empty:
    print(df_vel[df_vel.term == 'スキンケア'].head(10).to_string(index=False))


Trend velocity rows: 1500
 term week_start  interest  prev_week  wk_delta  rolling_avg
スキンケア 2020-01-01        79         77         2         72.3
スキンケア 2020-02-01        79         79         0         76.5
スキンケア 2020-03-01        81         79         2         79.0
スキンケア 2020-04-01        88         81         7         81.8
スキンケア 2020-05-01        99         88        11         86.8
スキンケア 2020-06-01        78         99       -21         86.5
スキンケア 2020-07-01        79         78         1         86.0
スキンケア 2020-08-01        77         79        -2         83.3
スキンケア 2020-09-01        79         77         2         78.3
スキンケア 2020-10-01        80         79         1         78.8


## 9. Reviewer Loyalty — First vs. Latest Review Gap

**SQL pattern:** `NTILE()` window function for quartile segmentation.
The gap between first and last review is a loyalty / engagement signal.
Heavy reviewers in the top quartile are the power users whose vocabulary
will anchor the NLP analysis in NB05.


In [10]:
sql_loyalty = '''
WITH reviewer_span AS (
    SELECT
        reviewer_id,
        source_id,
        COUNT(*)          AS total_reviews,
        MIN(review_date)  AS first_review,
        MAX(review_date)  AS last_review,
        CAST(
            (julianday(MAX(review_date)) - julianday(MIN(review_date)))
            AS INTEGER
        )                 AS days_active,
        COUNT(DISTINCT category_id) AS categories_reviewed
    FROM reviews
    WHERE reviewer_id IS NOT NULL
      AND review_date IS NOT NULL
    GROUP BY reviewer_id, source_id
    HAVING COUNT(*) >= 3
),
quartiles AS (
    SELECT
        source_id,
        NTILE(4) OVER (PARTITION BY source_id ORDER BY total_reviews) AS review_quartile,
        *
    FROM reviewer_span
)
SELECT
    source_id,
    review_quartile,
    COUNT(*)                           AS reviewer_count,
    ROUND(AVG(total_reviews), 1)       AS avg_reviews,
    ROUND(AVG(days_active), 0)         AS avg_days_active,
    ROUND(AVG(categories_reviewed), 2) AS avg_categories
FROM quartiles
GROUP BY source_id, review_quartile
ORDER BY source_id, review_quartile
'''

df_loyalty = pd.read_sql(sql_loyalty, conn)
print('Reviewer loyalty by quartile:')
print(df_loyalty.to_string(index=False))


Reviewer loyalty by quartile:
 source_id  review_quartile  reviewer_count  avg_reviews  avg_days_active  avg_categories
         2                1              48          3.0            575.0            2.73
         2                2              48          3.0            358.0            2.79
         2                3              48          3.0            434.0            2.60
         2                4              47          3.7            766.0            3.15
         3                1               4          3.0             45.0            0.00
         3                2               3          3.3            176.0            0.00
         3                3               3          4.0             74.0            0.00
         3                4               3         30.0            198.0            0.00


## 10. Summary Table — All Key Metrics

In [11]:
sql_summary = '''
SELECT 'Total reviews'         AS metric, CAST(COUNT(*) AS TEXT) AS value
FROM reviews
UNION ALL
SELECT 'Unique reviewers',
    CAST(COUNT(DISTINCT reviewer_id) AS TEXT)
FROM reviews WHERE reviewer_id IS NOT NULL
UNION ALL
SELECT 'Date range (reviews)',
    MIN(review_date) || ' to ' || MAX(review_date)
FROM reviews WHERE review_date IS NOT NULL
UNION ALL
SELECT 'Skincare reviews',
    CAST(COUNT(*) AS TEXT)
FROM reviews r
JOIN products p   ON r.product_id  = p.product_id
JOIN categories c ON p.category_id = c.category_id
WHERE COALESCE(p.tier_predicted, p.tier_override, c.tier) = 'skincare'
UNION ALL
SELECT 'Cosmetics reviews',
    CAST(COUNT(*) AS TEXT)
FROM reviews r
JOIN products p   ON r.product_id  = p.product_id
JOIN categories c ON p.category_id = c.category_id
WHERE COALESCE(p.tier_predicted, p.tier_override, c.tier) = 'cosmetics'
UNION ALL
SELECT 'Trend terms tracked',
    CAST(COUNT(DISTINCT term) AS TEXT)
FROM trends_weekly WHERE term_group = 'block_A'
UNION ALL
SELECT 'Trend weeks (block_A)',
    CAST(COUNT(DISTINCT week_start) AS TEXT)
FROM trends_weekly WHERE term_group = 'block_A'
UNION ALL
SELECT 'YouTube videos',
    CAST(COUNT(*) AS TEXT)
FROM yt_videos
UNION ALL
SELECT 'YouTube comments',
    CAST(COUNT(*) AS TEXT)
FROM yt_comments
UNION ALL
SELECT 'Rakuten products',
    CAST(COUNT(*) AS TEXT)
FROM products WHERE source_id = 1
UNION ALL
SELECT 'Weekly snapshot rows',
    CAST(COUNT(*) AS TEXT)
FROM products_weekly
UNION ALL
SELECT 'Snapshot dates',
    CAST(COUNT(DISTINCT snapshot_date) AS TEXT)
FROM products_weekly
'''

df_summary = pd.read_sql(sql_summary, conn)
print('SIGNAL/PULSE — DATABASE SUMMARY')
print('=' * 45)
for _, row in df_summary.iterrows():
    print(f'  {row["metric"]:<30} {row["value"]}')


SIGNAL/PULSE — DATABASE SUMMARY
  Total reviews                  22451
  Unique reviewers               20094
  Date range (reviews)           2005-01-07 to 2026-03-29
  Skincare reviews               17853
  Cosmetics reviews              3519
  Trend terms tracked            20
  Trend weeks (block_A)          87
  YouTube videos                 248
  YouTube comments               60676
  Rakuten products               31202
  Weekly snapshot rows           28907
  Snapshot dates                 1


In [12]:
conn.close()
print()
print('=' * 60)
print('NB03 — SQL Analytical Foundation: COMPLETE')
print('=' * 60)
print()
print('SQL patterns demonstrated:')
print('  CTEs, window functions (LAG, RANK, NTILE, SUM OVER)')
print('  Aggregate-first CTEs joined on key (Section 4)')
print('  EXISTS subquery for co-occurrence (Section 7)')
print('  NULLIF for safe division (Section 5)')
print('  Rolling averages and velocity (Section 8)')
print()
print('Proceed to NB04 — Consumer Voice (NLP)')



NB03 — SQL Analytical Foundation: COMPLETE

SQL patterns demonstrated:
  CTEs, window functions (LAG, RANK, NTILE, SUM OVER)
  Aggregate-first CTEs joined on key (Section 4)
  EXISTS subquery for co-occurrence (Section 7)
  NULLIF for safe division (Section 5)
  Rolling averages and velocity (Section 8)

Proceed to NB04 — Consumer Voice (NLP)
